# Computational Efficiency Comparison of CNN Models

This notebook evaluates the uploaded **Proposed CNN-SE** model against:

- MobileNetV2
- EfficientNetB0
- NASNetMobile

The comparison is performed under consistent inference conditions and reports:

1. Parameters (Millions)
2. FLOPs (G)
3. MACs (M)
4. Model Size (MB) — FP32 weight memory
5. Peak Runtime Memory (MB)
6. Latency (ms/image)
7. Throughput (images/s)

The notebook is designed to run on a local Windows/Linux/macOS machine in a TensorFlow/Keras environment.

> Important: latency and peak memory are hardware-dependent. The notebook records the CPU/GPU and software versions so the reported values are reproducible.


## 1. Install required packages

Run this cell once in your local environment.

If TensorFlow is already installed, you may skip the TensorFlow installation line.

For a CUDA-enabled NVIDIA GPU, install the TensorFlow version appropriate for your system. Do not blindly install CUDA packages if your existing environment is already configured and working.


In [ ]:
# Uncomment and run if required.
# %pip install -U tensorflow pandas numpy matplotlib psutil nbformat

# Optional FLOP-counting helper. The notebook has its own fallback FLOP counter,
# so this package is not mandatory.
# %pip install -U keras-flops


## 2. Imports and experiment configuration

In [ ]:
import os
import gc
import json
import time
import platform
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("CPU:", platform.processor())

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

# ------------------------------------------------------------
# EXPERIMENT CONFIGURATION
# ------------------------------------------------------------
# Put the .keras file in the same folder as this notebook,
# or change MODEL_PATH to its full local path.
MODEL_PATH = Path("best_CNN_SE_OCT8(1).keras")

INPUT_SIZE = (224, 224, 3)
NUM_CLASSES = 8
BATCH_SIZE = 1

# Number of warm-up and timed inference iterations.
WARMUP_STEPS = 20
TIMED_STEPS = 100

# Use float32 for a fair FP32 comparison.
DTYPE = tf.float32

# Baseline models are created without ImageNet weights because
# the comparison is computational/architectural, not an accuracy test.
# Change to "imagenet" only if you specifically want pretrained weights.
BASELINE_WEIGHTS = None

# Save all generated outputs here.
OUTPUT_DIR = Path("computational_efficiency_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nModel path:", MODEL_PATH.resolve())
print("Input:", INPUT_SIZE)
print("Batch size:", BATCH_SIZE)
print("Warm-up:", WARMUP_STEPS)
print("Timed iterations:", TIMED_STEPS)


## 3. Load the proposed CNN-SE model

The notebook first attempts to load the `.keras` file normally. If your model contains a custom unregistered layer, the cell prints the error and provides a place to add the corresponding custom layer class.

Because this is an efficiency experiment, the model is loaded with `compile=False`.


In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {MODEL_PATH.resolve()}\n"
        "Place the .keras model in the notebook folder or change MODEL_PATH."
    )

try:
    proposed_model = keras.models.load_model(
        MODEL_PATH,
        compile=False,
        safe_mode=False
    )
except Exception as e:
    print("Model loading failed.")
    print("Error:", repr(e))
    print("\nIf this is an 'Unknown layer' error, define your custom layer above")
    print("and pass it through custom_objects in keras.models.load_model().")
    raise

print(proposed_model.summary())
print("Input shape :", proposed_model.input_shape)
print("Output shape:", proposed_model.output_shape)
print("Parameters  :", proposed_model.count_params())


## 4. Build comparable baseline models

All baseline models use:

- 224 × 224 × 3 input
- `include_top=False`
- global average pooling
- an 8-class Dense output layer
- batch size 1 during measurement
- the same FP32 datatype

This makes the comparison structurally fair for an 8-class classification problem.

If your paper used a different input size, change `INPUT_SIZE` above and rerun the notebook.


In [ ]:
def build_mobilenetv2():
    base = keras.applications.MobileNetV2(
        include_top=False,
        weights=BASELINE_WEIGHTS,
        input_shape=INPUT_SIZE,
        pooling="avg"
    )
    x = layers.Dense(NUM_CLASSES, name="classifier")(base.output)
    return keras.Model(base.input, x, name="MobileNetV2_8Class")


def build_efficientnetb0():
    base = keras.applications.EfficientNetB0(
        include_top=False,
        weights=BASELINE_WEIGHTS,
        input_shape=INPUT_SIZE,
        pooling="avg"
    )
    x = layers.Dense(NUM_CLASSES, name="classifier")(base.output)
    return keras.Model(base.input, x, name="EfficientNetB0_8Class")


def build_nasnetmobile():
    base = keras.applications.NASNetMobile(
        include_top=False,
        weights=BASELINE_WEIGHTS,
        input_shape=INPUT_SIZE,
        pooling="avg"
    )
    x = layers.Dense(NUM_CLASSES, name="classifier")(base.output)
    return keras.Model(base.input, x, name="NASNetMobile_8Class")


# Build models one at a time later during memory measurement.
# We keep only the proposed model loaded initially.
model_builders = {
    "MobileNetV2": build_mobilenetv2,
    "EfficientNetB0": build_efficientnetb0,
    "NASNetMobile": build_nasnetmobile,
}

print("Baseline builders ready.")


## 5. Utility functions

The following functions calculate:

- parameter count
- FP32 weight memory
- FLOPs using TensorFlow graph profiling when available
- a transparent analytical fallback
- model inference latency
- throughput
- CPU process-memory change
- GPU peak memory when an NVIDIA/TF GPU is available


In [ ]:
def parameter_count_m(model):
    return model.count_params() / 1e6


def weight_memory_mb(model, dtype_bytes=4):
    """FP32 parameter memory; independent of .keras archive size."""
    return model.count_params() * dtype_bytes / (1024 ** 2)


def serialized_model_size_mb(model):
    """Save a temporary .keras archive and return its size."""
    tmp_dir = Path(tempfile.mkdtemp(prefix="efficiency_model_"))
    path = tmp_dir / "model.keras"
    try:
        model.save(path, include_optimizer=False)
        return path.stat().st_size / (1024 ** 2)
    finally:
        try:
            path.unlink(missing_ok=True)
            tmp_dir.rmdir()
        except Exception:
            pass


def _tensor_num_elements(shape):
    try:
        values = [int(x) for x in shape]
        if any(x is None for x in values):
            return None
        return int(np.prod(values))
    except Exception:
        return None


def analytical_core_flops(model, input_shape=INPUT_SIZE):
    """
    Fallback FLOP counter.

    Counts multiply/add operations for:
      Conv2D
      DepthwiseConv2D
      Dense

    FLOPs = 2 * MACs for these operations.

    Pooling, activations, batch normalization, reshaping, etc. are not
    included in this fallback. When TensorFlow's graph profiler works,
    the notebook uses the profiler result instead.
    """
    flops = 0
    details = []

    # Ensure a concrete input shape is available.
    dummy = tf.zeros((1, *input_shape), dtype=DTYPE)

    # Build all layers that have not yet been built.
    try:
        _ = model(dummy, training=False)
    except Exception:
        pass

    for layer in model.layers:
        try:
            if isinstance(layer, layers.Conv2D):
                kh, kw = layer.kernel_size
                in_ch = int(layer.input_shape[-1]) if hasattr(layer, "input_shape") else None
                if in_ch is None:
                    in_ch = int(layer.kernel.shape[-2])
                out_ch = int(layer.filters)
                out_shape = layer.output.shape
                h = out_shape[1]
                w = out_shape[2]
                if h is not None and w is not None:
                    macs = int(h) * int(w) * out_ch * kh * kw * in_ch
                    f = 2 * macs
                    flops += f
                    details.append((layer.name, "Conv2D", macs, f))

            elif isinstance(layer, layers.DepthwiseConv2D):
                kh, kw = layer.kernel_size
                in_ch = int(layer.depthwise_kernel.shape[-2])
                out_shape = layer.output.shape
                h = out_shape[1]
                w = out_shape[2]
                if h is not None and w is not None:
                    macs = int(h) * int(w) * in_ch * layer.depth_multiplier * kh * kw
                    f = 2 * macs
                    flops += f
                    details.append((layer.name, "DepthwiseConv2D", macs, f))

            elif isinstance(layer, layers.Dense):
                in_features = int(layer.kernel.shape[0])
                units = int(layer.units)
                macs = in_features * units
                f = 2 * macs
                flops += f
                details.append((layer.name, "Dense", macs, f))

        except Exception:
            continue

    return int(flops), pd.DataFrame(
        details, columns=["Layer", "Type", "MACs", "FLOPs"]
    )


def tensorflow_profiler_flops(model, input_shape=INPUT_SIZE):
    """
    Try TensorFlow's graph profiler.

    Returns:
        flops, method
    """
    try:
        @tf.function(
            input_signature=[
                tf.TensorSpec([1, *input_shape], dtype=tf.float32)
            ]
        )
        def forward(x):
            return model(x, training=False)

        concrete = forward.get_concrete_function()

        # Freeze variables so the profiler sees the full graph.
        from tensorflow.python.framework.convert_to_constants import (
            convert_variables_to_constants_v2_as_graph
        )

        frozen_func, graph_def = convert_variables_to_constants_v2_as_graph(
            concrete
        )

        with tf.Graph().as_default() as graph:
            tf.graph_util.import_graph_def(graph_def, name="")

            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            profile = tf.compat.v1.profiler.profile(
                graph=graph,
                options=opts
            )

            if profile is not None and profile.total_float_ops:
                return int(profile.total_float_ops), "TensorFlow graph profiler"

    except Exception as e:
        print("TensorFlow graph FLOP profiler unavailable:", type(e).__name__)

    return None, None


def calculate_flops(model):
    tf_flops, method = tensorflow_profiler_flops(model)

    if tf_flops is not None:
        return tf_flops, method

    fallback, _ = analytical_core_flops(model)
    return fallback, "Analytical Conv/DWConv/Dense fallback"


def reset_gpu_peak():
    if not tf.config.list_physical_devices("GPU"):
        return False
    try:
        for i in range(len(tf.config.list_physical_devices("GPU"))):
            tf.config.experimental.reset_memory_stats(f"GPU:{i}")
        return True
    except Exception:
        return False


def gpu_peak_memory_mb():
    if not tf.config.list_physical_devices("GPU"):
        return np.nan

    peaks = []
    for i in range(len(tf.config.list_physical_devices("GPU"))):
        try:
            info = tf.config.experimental.get_memory_info(f"GPU:{i}")
            peaks.append(info["peak"] / (1024 ** 2))
        except Exception:
            pass

    return max(peaks) if peaks else np.nan


def current_process_rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)


def benchmark_inference(model, warmup_steps=WARMUP_STEPS, timed_steps=TIMED_STEPS):
    """
    Measure batch-1 inference latency.

    The first call is excluded from timing because TensorFlow may perform
    tracing/JIT/kernel initialization.

    Returns mean latency and throughput.
    """
    x = tf.random.uniform(
        [BATCH_SIZE, *INPUT_SIZE],
        dtype=DTYPE,
        seed=123
    )

    # Trace/build.
    _ = model(x, training=False)

    # Warm-up.
    for _ in range(warmup_steps):
        y = model(x, training=False)
        _ = y.numpy()

    # Synchronize before timing.
    _ = model(x, training=False).numpy()

    times = []
    for _ in range(timed_steps):
        start = time.perf_counter()
        y = model(x, training=False)
        _ = y.numpy()  # synchronize for GPU
        end = time.perf_counter()
        times.append(end - start)

    times = np.asarray(times)
    mean_ms = float(times.mean() * 1000.0)
    std_ms = float(times.std(ddof=1) * 1000.0) if len(times) > 1 else 0.0
    throughput = 1000.0 / mean_ms

    return {
        "Latency (ms/image)": mean_ms,
        "Latency SD (ms)": std_ms,
        "Throughput (images/s)": throughput,
    }


def measure_peak_memory(model):
    """
    Measure memory while the model is built and executed.

    GPU:
        TensorFlow's peak allocator memory is reported.

    CPU:
        Process RSS before/after model execution is reported as a practical
        process-memory measure. It is not equivalent to TensorFlow allocator
        memory, so the CPU/GPU measurement method is explicitly recorded.
    """
    gc.collect()

    before_rss = current_process_rss_mb()
    reset_gpu_peak()

    x = tf.random.uniform([BATCH_SIZE, *INPUT_SIZE], dtype=DTYPE, seed=123)

    # Several calls ensure the graph is built and runtime buffers are allocated.
    for _ in range(10):
        _ = model(x, training=False).numpy()

    after_rss = current_process_rss_mb()
    gpu_peak = gpu_peak_memory_mb()

    return {
        "Peak GPU Memory (MB)": gpu_peak,
        "CPU RSS Increase (MB)": max(0.0, after_rss - before_rss),
        "Memory Measurement": (
            "TensorFlow GPU allocator peak" if not np.isnan(gpu_peak)
            else "Process RSS increase"
        )
    }


def summarize_model(model, name):
    print(f"\\n===== {name} =====")
    print("Input :", model.input_shape)
    print("Output:", model.output_shape)
    print("Params:", model.count_params())

    flops, method = calculate_flops(model)
    print("FLOPs :", flops, f"({flops/1e9:.6f} G)")
    print("Method:", method)

    return flops, method


## 6. Validate the proposed model input/output

The uploaded model should accept the same input resolution used for the baseline models.

If the model's actual input size is different, the notebook will stop rather than silently resize it.


In [ ]:
def shape_without_batch(shape):
    return tuple(int(x) for x in shape[1:] if x is not None)

proposed_input = shape_without_batch(proposed_model.input_shape)
proposed_output = shape_without_batch(proposed_model.output_shape)

print("Proposed model input :", proposed_input)
print("Configured input     :", INPUT_SIZE)
print("Proposed model output:", proposed_output)
print("Configured classes   :", NUM_CLASSES)

if proposed_input != tuple(INPUT_SIZE):
    print("\nWARNING: The uploaded model input does not match INPUT_SIZE.")
    print("Change INPUT_SIZE only if you intentionally want a different comparison.")

if proposed_output and proposed_output[-1] != NUM_CLASSES:
    print("\nWARNING: Uploaded model output classes do not match NUM_CLASSES.")
    print("For an architecture comparison, use the model's actual number of classes")
    print("or rebuild the baselines with the same output dimensionality.")


## 7. Calculate parameters, FLOPs and MACs

This section measures the architecture before the repeated latency/memory benchmark.

FLOP convention used by this notebook:

**1 MAC = 2 FLOPs** (one multiplication + one addition).

The exact FLOP count returned by TensorFlow's graph profiler is preferred. If it cannot profile a particular TensorFlow/Keras build, the notebook automatically falls back to an analytical Conv2D/DepthwiseConv2D/Dense count and records that method in the results.


In [ ]:
architectural_results = []

# Proposed model
flops, flop_method = calculate_flops(proposed_model)
architectural_results.append({
    "Model": "Proposed CNN-SE",
    "Parameters (M)": parameter_count_m(proposed_model),
    "FLOPs (G)": flops / 1e9,
    "MACs (M)": flops / 2e6,
    "Model Size (MB)": weight_memory_mb(proposed_model),
    "Serialized .keras Size (MB)": serialized_model_size_mb(proposed_model),
    "FLOP Method": flop_method,
})

# Baselines
baseline_models = {}
for model_name, builder in model_builders.items():
    print(f"Building {model_name}...")
    m = builder()
    baseline_models[model_name] = m

    flops, flop_method = calculate_flops(m)
    architectural_results.append({
        "Model": model_name,
        "Parameters (M)": parameter_count_m(m),
        "FLOPs (G)": flops / 1e9,
        "MACs (M)": flops / 2e6,
        "Model Size (MB)": weight_memory_mb(m),
        "Serialized .keras Size (MB)": serialized_model_size_mb(m),
        "FLOP Method": flop_method,
    })

architecture_df = pd.DataFrame(architectural_results)

display(
    architecture_df.style.format({
        "Parameters (M)": "{:.4f}",
        "FLOPs (G)": "{:.4f}",
        "MACs (M)": "{:.2f}",
        "Model Size (MB)": "{:.2f}",
        "Serialized .keras Size (MB)": "{:.2f}",
    })
)


## 8. Benchmark latency, throughput and peak memory

Important:

- Run this section on the same machine/device for all models.
- Close unnecessary GPU/CPU-heavy applications.
- Do not run another training job at the same time.
- For GPU measurements, TensorFlow may reserve memory; therefore GPU peak memory is reported using TensorFlow's allocator statistics.
- The notebook builds/benchmarks one model at a time to reduce interference.


In [ ]:
# Re-run the benchmark from clean model instances where practical.
# The proposed model remains loaded; baseline models are benchmarked sequentially.

benchmark_rows = []

def benchmark_one(model, name):
    print(f"\nBenchmarking: {name}")

    # Ensure variables are built.
    _ = model(tf.zeros([1, *INPUT_SIZE], dtype=DTYPE), training=False).numpy()

    mem = measure_peak_memory(model)
    timing = benchmark_inference(model)

    row = {
        "Model": name,
        **timing,
        **mem,
    }
    print(row)
    return row


benchmark_rows.append(benchmark_one(proposed_model, "Proposed CNN-SE"))

# Delete each baseline after measurement to minimize cross-model memory effects.
for name in ["MobileNetV2", "EfficientNetB0", "NASNetMobile"]:
    model = baseline_models[name]
    benchmark_rows.append(benchmark_one(model, name))

    del model
    gc.collect()
    tf.keras.backend.clear_session()

    # Rebuild the dictionary entry only if another cell needs it.
    baseline_models[name] = None

benchmark_df = pd.DataFrame(benchmark_rows)

display(
    benchmark_df.style.format({
        "Latency (ms/image)": "{:.3f}",
        "Latency SD (ms)": "{:.3f}",
        "Throughput (images/s)": "{:.2f}",
        "Peak GPU Memory (MB)": "{:.2f}",
        "CPU RSS Increase (MB)": "{:.2f}",
    })
)


## 9. Combine the final computational-efficiency table

The primary `Model Size (MB)` column is FP32 parameter memory:

`number of parameters × 4 bytes`

This is the most useful interpretation when a paper reports model size from parameter count.

The serialized `.keras` archive size is also retained separately because it includes file-format metadata and can differ substantially from weight memory.


In [ ]:
final_df = architecture_df.merge(
    benchmark_df,
    on="Model",
    how="left"
)

# Reorder columns to match the manuscript table.
final_columns = [
    "Model",
    "Parameters (M)",
    "FLOPs (G)",
    "MACs (M)",
    "Model Size (MB)",
    "Peak GPU Memory (MB)",
    "CPU RSS Increase (MB)",
    "Latency (ms/image)",
    "Throughput (images/s)",
    "Serialized .keras Size (MB)",
    "FLOP Method",
    "Memory Measurement",
]

final_df = final_df[final_columns]

display(
    final_df.style.format({
        "Parameters (M)": "{:.4f}",
        "FLOPs (G)": "{:.4f}",
        "MACs (M)": "{:.2f}",
        "Model Size (MB)": "{:.2f}",
        "Peak GPU Memory (MB)": "{:.2f}",
        "CPU RSS Increase (MB)": "{:.2f}",
        "Latency (ms/image)": "{:.3f}",
        "Throughput (images/s)": "{:.2f}",
        "Serialized .keras Size (MB)": "{:.2f}",
    })
)


## 10. Manuscript-style Table 5

For the paper, use one memory definition consistently.

If running on a GPU, the recommended manuscript table is:

- Parameters
- FLOPs
- MACs
- Model Size (FP32 weights)
- Peak GPU Memory
- Latency
- Throughput

If running on CPU, replace Peak GPU Memory with the CPU RSS increase and explicitly state the measurement method in the manuscript.


In [ ]:
paper_df = final_df.copy()

if paper_df["Peak GPU Memory (MB)"].notna().any():
    memory_col = "Peak GPU Memory (MB)"
    memory_header = "Peak Memory (MB)"
else:
    memory_col = "CPU RSS Increase (MB)"
    memory_header = "Peak Memory (MB)"

table5 = paper_df[[
    "Model",
    "Parameters (M)",
    "FLOPs (G)",
    "MACs (M)",
    "Model Size (MB)",
    memory_col,
    "Latency (ms/image)",
    "Throughput (images/s)"
]].rename(columns={memory_col: memory_header})

display(
    table5.style.format({
        "Parameters (M)": "{:.4f}",
        "FLOPs (G)": "{:.4f}",
        "MACs (M)": "{:.2f}",
        "Model Size (MB)": "{:.2f}",
        "Peak Memory (MB)": "{:.2f}",
        "Latency (ms/image)": "{:.2f}",
        "Throughput (images/s)": "{:.2f}",
    })
)


## 11. Relative efficiency compared with each baseline

This section calculates the percentage advantage of the Proposed CNN-SE.

For metrics where lower is better:

`Reduction (%) = (Baseline - Proposed) / Baseline × 100`

For throughput:

`Improvement (%) = (Proposed - Baseline) / Baseline × 100`


In [ ]:
proposed = final_df.loc[final_df["Model"] == "Proposed CNN-SE"].iloc[0]

comparison_rows = []

for _, row in final_df.iterrows():
    if row["Model"] == "Proposed CNN-SE":
        continue

    comparison_rows.append({
        "Baseline": row["Model"],
        "Parameter Reduction (%)":
            (row["Parameters (M)"] - proposed["Parameters (M)"])
            / row["Parameters (M)"] * 100,
        "FLOP Reduction (%)":
            (row["FLOPs (G)"] - proposed["FLOPs (G)"])
            / row["FLOPs (G)"] * 100,
        "MAC Reduction (%)":
            (row["MACs (M)"] - proposed["MACs (M)"])
            / row["MACs (M)"] * 100,
        "Model Size Reduction (%)":
            (row["Model Size (MB)"] - proposed["Model Size (MB)"])
            / row["Model Size (MB)"] * 100,
        "Latency Reduction (%)":
            (row["Latency (ms/image)"] - proposed["Latency (ms/image)"])
            / row["Latency (ms/image)"] * 100,
        "Throughput Improvement (%)":
            (proposed["Throughput (images/s)"] - row["Throughput (images/s)"])
            / row["Throughput (images/s)"] * 100,
    })

relative_df = pd.DataFrame(comparison_rows)

display(
    relative_df.style.format({
        "Parameter Reduction (%)": "{:.2f}",
        "FLOP Reduction (%)": "{:.2f}",
        "MAC Reduction (%)": "{:.2f}",
        "Model Size Reduction (%)": "{:.2f}",
        "Latency Reduction (%)": "{:.2f}",
        "Throughput Improvement (%)": "{:.2f}",
    })
)


## 12. Export results

The notebook saves both CSV and Excel-friendly files in:

`computational_efficiency_results/`

These files can be used to prepare the manuscript table.


In [ ]:
# Save CSV files.
table5.to_csv(OUTPUT_DIR / "Table5_Computational_Efficiency.csv", index=False)
final_df.to_csv(OUTPUT_DIR / "full_computational_efficiency_results.csv", index=False)
relative_df.to_csv(OUTPUT_DIR / "relative_efficiency_comparison.csv", index=False)

# Save an Excel workbook with separate sheets.
excel_path = OUTPUT_DIR / "Computational_Efficiency_Comparison.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    table5.to_excel(writer, sheet_name="Table5", index=False)
    final_df.to_excel(writer, sheet_name="Full Results", index=False)
    relative_df.to_excel(writer, sheet_name="Relative Comparison", index=False)

print("Saved:")
for p in [
    OUTPUT_DIR / "Table5_Computational_Efficiency.csv",
    OUTPUT_DIR / "full_computational_efficiency_results.csv",
    OUTPUT_DIR / "relative_efficiency_comparison.csv",
    excel_path,
]:
    print(" -", p.resolve())


## 13. Hardware/software record for the manuscript

Copy the output of the following cell into your experiment notes. This information is important because latency, throughput and runtime memory depend on the hardware/software environment.


In [ ]:
print("========== EXPERIMENT ENVIRONMENT ==========")
print("Date/time:", pd.Timestamp.now())
print("OS:", platform.platform())
print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("CPU:", platform.processor())
print("Machine:", platform.machine())
print("Logical CPUs:", os.cpu_count())
print("GPUs:", tf.config.list_physical_devices("GPU"))

if gpus:
    for i, gpu in enumerate(gpus):
        try:
            details = tf.config.experimental.get_device_details(gpu)
        except Exception:
            details = {}
        print(f"GPU {i}:", gpu)
        print("GPU details:", details)

print("Input:", INPUT_SIZE)
print("Batch size:", BATCH_SIZE)
print("Precision:", DTYPE.name)
print("Warm-up iterations:", WARMUP_STEPS)
print("Timed iterations:", TIMED_STEPS)
print("Baseline weights:", BASELINE_WEIGHTS)


## 14. Optional: save the final table as a publication-ready image

This cell creates a PNG version of Table 5. It is optional and does not affect the numerical results.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15, 3.5))
ax.axis("off")

display_df = table5.copy()
display_df["Parameters (M)"] = display_df["Parameters (M)"].map(lambda x: f"{x:.4f}")
display_df["FLOPs (G)"] = display_df["FLOPs (G)"].map(lambda x: f"{x:.4f}")
display_df["MACs (M)"] = display_df["MACs (M)"].map(lambda x: f"{x:.2f}")
display_df["Model Size (MB)"] = display_df["Model Size (MB)"].map(lambda x: f"{x:.2f}")
display_df["Peak Memory (MB)"] = display_df["Peak Memory (MB)"].map(lambda x: f"{x:.2f}")
display_df["Latency (ms/image)"] = display_df["Latency (ms/image)"].map(lambda x: f"{x:.2f}")
display_df["Throughput (images/s)"] = display_df["Throughput (images/s)"].map(lambda x: f"{x:.2f}")

table_artist = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    cellLoc="center",
    loc="center"
)

table_artist.auto_set_font_size(False)
table_artist.set_fontsize(9)
table_artist.scale(1, 1.6)

plt.tight_layout()
png_path = OUTPUT_DIR / "Table5_Computational_Efficiency.png"
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", png_path.resolve())


# Interpretation notes

### Model Size
The manuscript table uses FP32 parameter memory rather than the `.keras` archive size. For example:

`Model Size = Parameters × 4 bytes / 1024²`

### FLOPs and MACs
This notebook uses the convention:

`FLOPs = 2 × MACs`

The FLOP-counting method is recorded automatically.

### Latency
Latency is measured with batch size 1 after warm-up. The notebook synchronizes GPU results by converting the output tensor to NumPy before stopping the timer.

### Throughput
For batch size 1:

`Throughput = 1000 / Latency(ms/image)`

### Reproducibility
For your manuscript, report the CPU/GPU, TensorFlow/Keras version, input size, batch size, precision, warm-up iterations, and timing iterations.

Do not compare latency numbers obtained on different hardware as though they were directly comparable.
